# 00 — Google Colab setup

This notebook prepares the existing SAR-MIGRES repository for Colab without changing its repository structure. It clones the project, optionally mounts Google Drive for persistent data/output/cache, installs the project, clones the pinned upstream MIGRES implementation into `src/MIGRES`, verifies the expected commit/file hashes, checks the GPU and scientific dependencies, and validates the canonical configuration.

Run this notebook before the existing experiment notebooks.

In [ ]:
# ===== USER SETTINGS =====
PROJECT_REPO_URL = "https://github.com/KunalDahal/State-Aware-Retrieval.git"
PROJECT_BRANCH = "master"
PROJECT_DIR = "/content/SAR"
MIGRES_URL = "https://github.com/AdelWang/MIGRES.git"
MIGRES_DIR = f"{PROJECT_DIR}/src/MIGRES"
MOUNT_GOOGLE_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/SAR-MIGRES"
INSTALL_EXTRAS = "all"  # use "all" for the real research environment
SET_GEMINI_KEY_FROM_COLAB_SECRET = True

# Do not change these for the canonical scientific protocol.
EXPECTED_MIGRES_COMMIT = "1a67b8769ca3fdf2b497e109e529143822248ec5"
EXPECTED_SAR_K = 50
EXPECTED_DATASETS = ("hotpotqa", "musique", "natural_questions")


In [ ]:
# ===== CLONE / SELECT THE PROJECT =====
from pathlib import Path
import os, subprocess, sys

project = Path(PROJECT_DIR)
if not (project / "pyproject.toml").exists():
    if not PROJECT_REPO_URL:
        raise ValueError("Set PROJECT_REPO_URL in the cell above before the first Colab run.")
    if project.exists():
        raise RuntimeError(f"{PROJECT_DIR} exists but is not the expected project checkout. Remove it or choose another PROJECT_DIR.")
    subprocess.run(["git", "clone", "--branch", PROJECT_BRANCH, PROJECT_REPO_URL, PROJECT_DIR], check=True)
os.chdir(project)
print("Project:", project)
print("Git:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
# ===== GOOGLE DRIVE FOR PERSISTENT DATA / OUTPUTS =====
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    drive_project = Path(DRIVE_PROJECT_DIR)
    drive_project.mkdir(parents=True, exist_ok=True)
    # Keep code in /content/SAR; persist only large/generated runtime state.
    for name in ("data", "outputs"):
        src = project / name
        dst = drive_project / name
        dst.mkdir(parents=True, exist_ok=True)
        if src.is_symlink() and src.resolve() == dst.resolve():
            continue
        if src.exists() and not src.is_symlink():
            # Preserve any existing local content before replacing the directory with a symlink.
            for item in src.iterdir():
                target = dst / item.name
                if not target.exists():
                    item.rename(target)
            src.rmdir()
        if not src.exists():
            src.symlink_to(dst, target_is_directory=True)
    print("Persistent runtime directories:", drive_project)
else:
    print("Google Drive mounting disabled. Colab runtime storage is ephemeral.")


In [ ]:
# ===== INSTALL PROJECT + RESEARCH DEPENDENCIES =====
subprocess.run([sys.executable, "-m", "pip", "install", "-e", f".[{INSTALL_EXTRAS}]"], check=True)
print("Installed SAR-MIGRES from", project)


In [ ]:
# ===== CLONE / REPLACE MIGRES AT src/MIGRES =====
migres = Path(MIGRES_DIR)
if not (migres / ".git").exists():
    if migres.exists() and any(migres.iterdir()):
        raise RuntimeError(f"{MIGRES_DIR} exists but is not an empty git checkout. Remove it or change MIGRES_DIR.")
    subprocess.run(["git", "clone", MIGRES_URL, MIGRES_DIR], check=True)
subprocess.run(["git", "-C", MIGRES_DIR, "fetch", "--all", "--tags", "--prune"], check=True)
subprocess.run(["git", "-C", MIGRES_DIR, "checkout", "--detach", EXPECTED_MIGRES_COMMIT], check=True)
actual_commit = subprocess.check_output(["git", "-C", MIGRES_DIR, "rev-parse", "HEAD"], text=True).strip()
if actual_commit != EXPECTED_MIGRES_COMMIT:
    raise RuntimeError(f"MIGRES commit mismatch: expected {EXPECTED_MIGRES_COMMIT}, got {actual_commit}")
os.environ["MIGRES_REPO_DIR"] = str(migres)
print("MIGRES:", migres)
print("MIGRES commit:", actual_commit)


In [ ]:
# ===== VERIFY MIGRES FILE HASHES AGAINST THE ADAPTER =====
from sar_migres.migres import adapter
from sar_migres.utils.hashing import sha256_file

if actual_commit != adapter.TESTED_UPSTREAM_COMMIT:
    raise RuntimeError("MIGRES commit does not match the commit pinned by sar_migres.migres.adapter")
for rel, expected in adapter.TESTED_UPSTREAM_SHA256.items():
    path = migres / rel
    if not path.exists():
        raise FileNotFoundError(f"Required MIGRES file missing: {path}")
    got = sha256_file(path)
    if got != expected:
        raise RuntimeError(f"MIGRES hash mismatch for {rel}: expected {expected}, got {got}")
    print(f"OK  {rel}")


In [ ]:
# ===== CONFIGURE GEMINI THROUGH COLAB SECRETS =====
if SET_GEMINI_KEY_FROM_COLAB_SECRET:
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key:
            os.environ["GEMINI_API_KEY"] = key
            print("GEMINI_API_KEY loaded from Colab Secret.")
        else:
            print("GEMINI_API_KEY secret is empty; real LLM runs will fail until it is set.")
    except Exception as exc:
        print(f"GEMINI_API_KEY not loaded from Colab Secret: {exc}")
os.environ.setdefault("OPENAI_BASE_URL", "https://generativelanguage.googleapis.com/v1beta/openai/")


In [ ]:
# ===== ENVIRONMENT / GPU CHECK =====
import importlib.util, platform
print("Python:", platform.python_version())
required = ["numpy", "yaml", "torch", "transformers", "sentence_transformers", "bayes_opt", "openai"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
print("Missing optional scientific modules:", missing or "none")
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as exc:
    print("Torch check failed:", exc)


In [ ]:
# ===== CANONICAL SCIENTIFIC PROTOCOL CHECK =====
from sar_migres.utils.config import load_config, validate_canonical_protocol
cfg = load_config("src/configs/default.yaml", "src/configs/gemini.yaml")
validate_canonical_protocol(cfg)
if tuple(cfg.data.datasets) != EXPECTED_DATASETS:
    raise RuntimeError(f"Dataset mismatch: expected {EXPECTED_DATASETS}, got {tuple(cfg.data.datasets)}")
if cfg.sar.k != EXPECTED_SAR_K:
    raise RuntimeError(f"K mismatch: expected {EXPECTED_SAR_K}, got {cfg.sar.k}")
cfg.migres.repo_dir = str(migres)
print("Canonical protocol: OK")
print("Datasets:", cfg.data.datasets)
print("K:", cfg.sar.k)
print("Chunking:", cfg.data.corpus_chunk_max_tokens, "tokens /", cfg.data.corpus_chunk_overlap_tokens, "overlap")
print("Final labeled split:", cfg.data.final_split)
print("BO trials:", 3 + 7)


In [ ]:
# ===== PACKAGE / CLI SMOKE CHECK =====
subprocess.run([sys.executable, "-m", "compileall", "-q", "src"], check=True)
for mod in [
    "sar_migres.corpus.builder", "sar_migres.datasets.loader", "sar_migres.embeddings.encoder",
    "sar_migres.scoring.sar", "sar_migres.retrieval.sar", "sar_migres.retrieval.baselines",
    "sar_migres.migres.adapter", "sar_migres.experiments.runner", "sar_migres.evaluation.retrieval",
]:
    __import__(mod)
print("Package import/compile check: OK")
print("Next: open 01_parameter_hyperparameter_selection.ipynb and run the canonical experiment flow.")
